# 02. How far can we trust a score?

**Goal:** `01` ended with one number for one model. Before we use numbers like that to choose between models, we need to know how much of the number is real and how much is luck.

Two kinds of luck can move a score:

1. **Which videos ended up in the test set.** A different handful of videos would give a different score.
2. **Which random numbers training used.** Training the same model twice gives two slightly different models.

**What you'll learn:**
1. Why few videos means a shaky score
2. What a bootstrap and a 95% interval are
3. What a seed is, and why one run isn't enough
4. How big an improvement has to be before we believe it

Run `01` first, because it builds the crops. This notebook trains 5 models at about 8 minutes each on an Apple GPU, so expect about 40 minutes.


## 1. Setup

The shared code and settings are in [brackish.py](brackish.py). A **seed** is the starting value for a program's random numbers. The same seed gives the same random choices, and a different seed gives different ones.


In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from brackish import (
    CLASSES, CROPS, SEED, bootstrap, confusion, interval, metrics, predict,
    report_interval, test_videos, train, videos_per_class,
)

SEEDS = [SEED, 1, 2, 3, 4]

if not (CROPS / "by_video").exists():
    raise FileNotFoundError("Run 01_brackish_classification.ipynb first. It builds the crops.")


## 2. Count the videos, not the crops

Crops from one video are close to copies of each other: same water, same light, often the same animal in neighbouring frames. So 56 shrimp crops from 2 videos hold far less information than 56 crops from 56 videos.

Read this table for the classes with low video counts. Their scores rest on the least evidence.


In [ ]:
videos_per_class("by_video")


## 3. One model, and how much its score could move

We can't film new test videos, but we can imitate it. A **bootstrap** builds a new test set by drawing videos at random from the real test set, **with replacement**: after a video is drawn it goes back in the pool, so one video can appear twice and another not at all. We score the model on that new set, then repeat 1,000 times.

We draw whole videos, because crops from one video stick together. Drawing single crops would pretend we had far more independent evidence than we do.

The **95% interval** is the range holding the middle 95% of those 1,000 scores. A wide interval means the score depends heavily on which videos we got.

The last column of the table shows how many redrawn test sets had no example of that class. Those rounds are left out of that class's interval and its macro F1.


In [ ]:
model, loaders, history = train("by_video", seed=SEED)
truth, guesses = predict(model, loaders["test"])
videos = test_videos(loaders)

score = metrics(confusion(truth, guesses))
drawn = bootstrap(truth, guesses, videos)
report_interval(score, drawn)


In [ ]:
low, high = interval(drawn["macro_f1"])
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(drawn["macro_f1"], bins=30, color="#1f5eff")
ax.axvline(score["macro_f1"], color="#111827", linewidth=2, label="score on the real test set")
ax.axvspan(low, high, color="#1f5eff", alpha=0.1, label="95% interval")
ax.set_title("Macro F1 on 1,000 redrawn test sets")
ax.set_xlabel("macro F1")
ax.set_ylabel("redrawn test sets")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()


## 4. Same model, different seeds

Now the second kind of luck. Random numbers decide how weights start in the new last layer, which crops are capped, what order the images come in, and which flips and colour changes they get.

We train the same model with 5 seeds and score each on the same test set. Nothing about the model or the data has changed, so any difference between these runs is luck. The seed in section 3 counts as the first run.


In [ ]:
runs = {SEED: (truth, guesses)}
for seed in tqdm(SEEDS[1:], desc="seeds"):
    seed_model, seed_loaders, _ = train("by_video", seed=seed, live=False)
    runs[seed] = predict(seed_model, seed_loaders["test"])
scores = {seed: metrics(confusion(*run)) for seed, run in runs.items()}

print(f"{'seed':>4s}{'accuracy':>10s}{'macro F1':>10s}" + "".join(f"{name:>12s}" for name in CLASSES))
for seed, result in scores.items():
    per_class = "".join(f"{value:12.2f}" for value in result["f1"])
    print(f"{seed:4d}{result['accuracy']:10.3f}{result['macro_f1']:10.3f}{per_class}")
macro = [result["macro_f1"] for result in scores.values()]
print(f"\nMacro F1 across seeds: mean {np.mean(macro):.3f}, lowest {min(macro):.3f}, highest {max(macro):.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for row, name in enumerate(CLASSES):
    ax.scatter([result["f1"][row] for result in scores.values()], [row] * len(scores),
               s=64, color="#1f5eff", alpha=0.6, linewidths=0)
ax.set_yticks(range(len(CLASSES)), CLASSES)
ax.set_xlim(0, 1)
ax.invert_yaxis()
ax.set_xlabel("F1 on the test set")
ax.set_title("Each dot is one seed")
ax.spines[["top", "right"]].set_visible(False)
plt.show()


## 5. The number to quote

Both kinds of luck happen at once, so we combine them: bootstrap the test set for every seed, then pool all the redrawn scores. This is the honest way to report the model: a mean with its interval, not one lucky run.


In [ ]:
draws = {seed: bootstrap(*run, videos) for seed, run in tqdm(runs.items(), desc="seeds")}
pooled = np.concatenate([draw["macro_f1"] for draw in draws.values()])
low, high = interval(pooled)
print(f"ResNet18, split by video: macro F1 {np.mean(macro):.3f}, 95% interval {low:.3f} to {high:.3f}")


## 6. How big must an improvement be?

Later we'll compare new ideas against this model. To compare two models fairly, we score them on the same redrawn test sets and look at the **difference** in each round. Hard videos are hard for both, so this cancels out much of the test-set luck.

Here's a useful trick: compare the model with itself. Every pair of seeds is the same model, so the true difference is zero. Whatever differences show up are the size of pure luck. A new idea has to beat this.


In [ ]:
differences = np.concatenate([
    draws[a]["macro_f1"] - draws[b]["macro_f1"] for a, b in itertools.permutations(SEEDS, 2)
])
low, high = interval(differences)
print(f"Between seeds of the same model, macro F1 differs by {low:+.3f} to {high:+.3f}")
print(f"Treat an improvement under {max(-low, high):.3f} as luck")


## 7. What to take away

Answer these from your own output:

1. Which class has the widest interval? How many test videos does it have?
2. Which moves macro F1 more: the choice of test videos or the seed?
3. Could a model that scores 0.02 higher than this one really be better?

**Remember the limit:** a bootstrap can only redraw the videos we have. If a class has 2 test videos, redrawing them can't show how the model does on a third kind of video, so its real uncertainty is larger than its interval. And none of this says how the model does in Nova Scotia.

**Next:** with this ruler in place, we can try stronger models and fairly compare them. `03` moves from classification to detection.
